# Train a ukiyo-e LoRA on FLUX 3 Image Klein

A LoRA is a small add-on that steers an image model toward a style. This is the notebook from my LoRA video: 246 color ukiyo-e prints, the trigger word `ukiyocolor`, and AI Toolkit. Swap in your own images whenever you like.

## What you need

A Linux machine with a GPU that has at least 24 GB of VRAM, [AI Toolkit](https://github.com/ostris/ai-toolkit), and access to [FLUX 3 Image Klein base](https://huggingface.co/black-forest-labs/flux-3-image-klein-base) on Hugging Face. Get access before logging in. You'll also need Git, Python, and a CUDA-compatible PyTorch build for your driver.

No GPU like that? Rent one. On [RunPod](https://www.runpod.io/), start a pod with 24 GB of VRAM or more from a PyTorch template, open JupyterLab and upload this notebook. Files under `/workspace` stay when the pod stops, so the setup below works there if it exists, and in your home folder otherwise.

In [ ]:
from pathlib import Path
import os
import subprocess

root = Path("/workspace") if Path("/workspace").is_dir() else Path.home()
work = root / "klein-ukiyocolor"
work.mkdir(parents=True, exist_ok=True)
toolkit = work / "ai-toolkit"
repo = work / "flux3-klein-lora-guide"
output = work / "output"
python = toolkit / "venv/bin/python"
print(work)

In [ ]:
subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", str(toolkit)], check=True)
subprocess.run(["git", "checkout", "04733873cf1da8060fce01316ec45daedbd4a918"], cwd=toolkit, check=True)
subprocess.run(["python3", "-m", "venv", str(toolkit / "venv")], check=True)

I used PyTorch 2.13.0 for CUDA 13.0. Install a CUDA build that fits your driver first, then install AI Toolkit's requirements. The check should print `True`.

In [ ]:
subprocess.run([str(python), "-m", "pip", "install", "torch==2.13.0", "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-r", "requirements.txt"], cwd=toolkit, check=True)
subprocess.run([str(python), "-c", "import torch; print(torch.cuda.is_available())"], check=True)

Log in to Hugging Face with an account that has access to the model. If you can't type into the login prompt here, run `ai-toolkit/venv/bin/hf auth login` in a terminal from the work folder.

In [ ]:
!{toolkit}/venv/bin/hf auth login

## 1. The data

I used 246 color ukiyo-e prints from the [Cleveland Museum of Art Open Access](https://www.clevelandart.org/open-access) collection, all CC0. `metadata.csv` keeps each print's source page and license, which matters if you share your LoRA.

Look over your data before you train. Take out duplicates, bad scans, and images that don't fit the look, like black and white prints in a color set. Your images can be different sizes and shapes.

In [ ]:
subprocess.run(["git", "clone", "https://github.com/MaanavD/flux3-klein-lora-guide.git", str(repo)], check=True)
data = repo / "ukiyoe_color"
images = sorted(data.glob("*.jpg"))
captions = sorted(data.glob("*.txt"))
print(f"{len(images)} images, {len(captions)} captions")
print("Images without captions:", [p.name for p in images if not p.with_suffix(".txt").exists()])

## 2. Captions and the trigger word

Each image has a `.txt` file with the same name. Describe what's in the picture: people, objects, setting and any writing. Leave out the style, like "woodblock print" or "ukiyo-e", because that's what the LoRA learns. An LLM wrote these captions, so read some and fix missing subjects or wrong details.

Every caption starts with `ukiyocolor`. The style gets tied to that word, and you put it in your prompt to turn the LoRA on. A short made-up word like `vqx` carries no meaning of its own. A word like `ukiyocolor` is easier to remember, but the model may already read something into it, so check what it does on the base model first. The smoke test below does that for you: AI Toolkit samples before the first training step, so step 0 shows the base model with and without the trigger.

In [ ]:
print((data / "ukiyoe_color_0002.txt").read_text())

## 3. The recipe

This is the same recipe as my Halloween LoRA. Most of AI Toolkit's defaults work great.

- Rank sets how much the LoRA can hold. Think of the base model as a textbook and the LoRA as sticky notes: a small rank is enough for a few pointers, and a whole art style can use more. Rank 32 is about 80 million weights, a 161 MB file.
- Learning rate is how much the LoRA changes each step. I kept it at 1e-4 with a `cosine` schedule, which starts high and ends low. `constant_with_warmup` is the other one I liked.
- `timestep_type: shift` practices on high noise about as often as Klein's sampler sees it.
- 4,000 steps, saving and sampling every 100, with seed 42, so you can compare checkpoints.

The rest are AI Toolkit defaults or settings that keep memory down: AdamW, batch size 1, 1024 px, bf16, gradient checkpointing and latents cached to disk. This copies `configs/ukiyocolor.yaml` and fills in the two paths. The original stays untouched.

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "PyYAML"], check=True)
import yaml

config = yaml.safe_load((repo / "configs/ukiyocolor.yaml").read_text())
process = config["config"]["process"][0]
process["training_folder"] = str(output)
process["datasets"][0]["folder_path"] = str(data)
output.mkdir(parents=True, exist_ok=True)
config_path = work / "ukiyocolor.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 4. Train and watch the samples

First, a 50-step smoke test. It saves and samples every 25 steps and has its own run name, so its checkpoints stay separate from the full run.

In [ ]:
import copy
smoke = copy.deepcopy(config)
smoke["config"]["name"] += "-smoke"
smoke_process = smoke["config"]["process"][0]
smoke_process["train"]["steps"] = 50
smoke_process["save"]["save_every"] = 25
smoke_process["sample"]["sample_every"] = 25
smoke_path = work / "ukiyocolor-smoke.yaml"
smoke_path.write_text(yaml.safe_dump(smoke, sort_keys=False))
subprocess.run([str(python), "run.py", str(smoke_path)], cwd=toolkit, check=True)

If the smoke test works, start the full run. The sample prompts are your test questions: a tiger with and without `ukiyocolor`, and an astronaut on the moon, which no ukiyo-e artist ever drew. The tiger without the trigger should stay close to the base model the whole time. The same seed lets you compare checkpoints fairly.

In [ ]:
subprocess.run([str(python), "run.py", str(config_path)], cwd=toolkit, check=True)

Open the checkpoints and `samples/` in the run folder and judge them by eye.

In [ ]:
run_dir = output / config["config"]["name"]
print("Run folder:", run_dir)
print("Samples:", run_dir / "samples")
print("Checkpoints:", *sorted(run_dir.glob("*.safetensors")), sep="\n")

## 5. Pick a checkpoint

Compare the same prompts and seeds across checkpoints. Early ones still look too real. Late ones can learn the training images too well and start copying them instead of the style. You don't have to pick the last checkpoint just because it trained the longest. Pick the one that looks the way you want, and figure out why you like it.

If none of them get there, change one thing and train again: steps, learning rate, schedule, or your captions and prompts.

## 6. Use it

Pick a checkpoint from your run. Put `ukiyocolor` in the prompt and describe anything you want. I used strength 1. Lower it to make the style softer.

In [ ]:
# Use your own step 2,000 checkpoint, or change the path to your favorite.
checkpoint = run_dir / f"{config['config']['name']}_000002000.safetensors"
assert checkpoint.is_file(), checkpoint
print(checkpoint)

In [ ]:
prompt = "ukiyocolor, an astronaut walking on the moon with earth in the sky"
env = os.environ.copy()
env["PYTHONPATH"] = str(toolkit) + os.pathsep + env.get("PYTHONPATH", "")
image = work / "astronaut.jpg"
subprocess.run([str(python), str(repo / "scripts/make_card.py"), "--lora", str(checkpoint),
                "--prompt", prompt, "--out", str(image), "--seed", "42", "--strength", "1"],
               cwd=toolkit, env=env, check=True)
print(image)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(image)))

## Make your own

Now swap in images you care about: an old camcorder's footage, a favorite animated world, your own drawings. Gather them, record their sources and licenses, caption what's pictured, pick a trigger word, train, and compare checkpoints. Museum open access collections like the [Cleveland Museum of Art](https://www.clevelandart.org/open-access) and the [Art Institute of Chicago](https://www.artic.edu/collection), [The Public Domain Review](https://publicdomainreview.org/) and Wikimedia Commons are good places to start. Use them responsibly, and have fun.